## **Eval of Finetuned Qwen3-8B with a Cross-Encoder**


In [8]:
%pip -q install transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 22.5 MB/s eta 0:00:00


Miscellaneous

In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

Define a function to load the two models (base and finetuned). Run one after the other.

In [10]:
def load_model(model_name):
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        quantization_config=quant_config,
    )
    return tokenizer, model

Define function to prompt

In [11]:
def prompt_model(tokenizer, model, prompt, max_new_tokens=500):
    messages = [
        {"role": "system", "content": "Rewrite the user's text in Gen-Z brainrot slang. Keep the meaning."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
        )

    generated = outputs[0][inputs.input_ids.shape[1]:]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

In [12]:
base_tokenizer, base_model = load_model("Qwen/Qwen3-4B")
s = prompt_model(
    base_tokenizer,
    base_model,
    "What is the meaning of life?"
)
print(type(s))

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

<class 'str'>


Setting up evaluator (cross encoder)

In [3]:
from sentence_transformers import util, CrossEncoder # SentenceTransformer

# evaluator = SentenceTransformer("all-MiniLM-L6-v2")
evaluator = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")

def semantic_score(reference, prediction):
    score = evaluator.predict([reference, prediction])
    return score.item()

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Read test lines

In [24]:
import json
data = []
with open("test.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data.append(json.loads(line))

0.1348426342010498


Run base and finetuned model on test prompts and compare to test answers

In [ ]:
base_tokenizer, base_model = load_model("Qwen/Qwen3-4B")
base_answers = [prompt_model(
    base_tokenizer,
    base_model,
    item["source"]
) for item in data]

In [ ]:
finetuned_tokenizer, finetuned_model = load_model("FINETUNED_MODEL")
finetuned_answers = [prompt_model(
    finetuned_tokenizer,
    finetuned_model,
    item["source"]
) for item in data]

In [ ]:
base_scores = [semantic_score(
    base_answers[i], data[i]["target"]
) for i in range(len(data))]
finetuned_scores = [semantic_score(
    finetuned_answers[i], data[i]["target"]
) for i in range(len(data))]

